# SABO Graph Statistics — Exploratory Visualisation
**Study #1** · Multi-language empirical characterisation of software structure

This notebook loads `repo_stats.csv` and `repo_edge_label_matrix.csv` produced
by `compute_graph_stats.py` and produces exploratory visualisations for Java vs C#
comparison.  All cells are self-contained and re-runnable.

> **Path setup**: edit the two variables in the *Configuration* cell below, then
> *Run All*.  Stats files for different languages can live in separate directories
> or be merged; the loader handles both cases.


## 0 · Dependencies

Loads the standard PyData stack (pandas, numpy, matplotlib, seaborn, scipy.stats) plus a fixed seaborn theme and a per-language colour palette so every figure later in the notebook is visually consistent. The `log1p` transform, Spearman correlations, and Mann–Whitney U test all live in this stack — the imports here decide what statistical machinery the rest of the notebook can reach for.

**References:**

- McKinney 2010, *Data Structures for Statistical Computing in Python* (Proc. 9th Python in Science Conf., 56–61)
- Hunter 2007, *Matplotlib: A 2D Graphics Environment* (Computing in Science & Engineering 9(3):90–95)
- Waskom 2021, *seaborn: statistical data visualization* (Journal of Open Source Software 6(60):3021)

In [ ]:
# Standard-library + typical data-science stack.
# Uncomment the pip line if any package is missing.
# %pip install pandas matplotlib seaborn scipy

import warnings, itertools, textwrap
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats as spstats
from pathlib import Path

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid", palette="muted", font="Noto Sans", font_scale=1.05)
LANG_PALETTE = {"java": "#4C72B0", "csharp": "#DD8452", "cpp": "#55A868"}


## 1 · Configuration & data loading

Reads the per-repo summary (`repo_stats.csv`) and the structural edge-label tally (`repo_edge_label_matrix.csv`) from each language directory, force-tags the `lang` column, concatenates the frames, and coerces all non-key columns to numeric. This mirrors the standard MSR pipeline (mine → extract → aggregate → analyse); printing the row counts at the end is a sanity check that no language was silently skipped because of a missing file.

**References:**

- Hassan 2008, *The Road Ahead for Mining Software Repositories* (FoSM 2008, pp. 48–57)
- Kalliamvakou et al. 2016, *An in-depth study of the promises and perils of mining GitHub* (Empirical Software Engineering 21(5):2035–2071) — for why provenance and manifest checks matter when working from extracted CSVs

In [ ]:
# ── Edit these paths ──────────────────────────────────────────────────────────
# Each entry is a directory that contains repo_stats.csv and
# repo_edge_label_matrix.csv.  You can point multiple languages at different
# directories; the loader will concatenate them.
STATS_DIRS = {
    "java":   Path("stats/java_v3.1"),
    "csharp": Path("stats/csharp"),
    # "cpp": Path("stats/cpp"),
}
# ─────────────────────────────────────────────────────────────────────────────

def load_csv(dirs: dict, filename: str) -> pd.DataFrame:
    frames = []
    for lang, d in dirs.items():
        p = Path(d) / filename
        if not p.exists():
            print(f"[WARN] {p} not found — skipping")
            continue
        df = pd.read_csv(p, dtype={"subrepo": str})
        # Ensure lang column matches the key even if the file disagrees.
        df["lang"] = lang
        frames.append(df)
    if not frames:
        raise FileNotFoundError("No stats files found. Check STATS_DIRS.")
    return pd.concat(frames, ignore_index=True)

stats  = load_csv(STATS_DIRS, "repo_stats.csv")
matrix = load_csv(STATS_DIRS, "repo_edge_label_matrix.csv")

# Coerce numeric columns (empty strings become NaN).
for df in (stats, matrix):
    for col in df.columns:
        if col not in ("repo", "subrepo", "lang"):
            df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"stats  : {stats.shape[0]:,} rows × {stats.shape[1]} cols")
print(f"matrix : {matrix.shape[0]:,} rows × {matrix.shape[1]} cols")
print()
print(stats.groupby("lang").size().rename("repos").to_frame())


## 2 · Per-language descriptive summary

Computes median, mean, and standard deviation for every node-count (`n_*`) and edge-count (`e_*`) column, split by language. Software-engineering counts are almost always right-skewed, so the *median* is the honest centre and the gap between median and mean is itself a quick diagnostic of skew. This is the obligatory "look at the data first" step that any inferential analysis below must rest on.

**References:**

- Fenton & Bieman 2014, *Software Metrics: A Rigorous and Practical Approach* (3rd ed., CRC Press) — for the standard descriptive battery on software counts
- Tukey 1977, *Exploratory Data Analysis* (Addison-Wesley)

In [ ]:
# Numeric summary split by language.
NODE_COLS = [c for c in stats.columns if c.startswith("n_") and c != "n_project"]
EDGE_COLS = [c for c in stats.columns if c.startswith("e_")]

summary = (
    stats.groupby("lang")[NODE_COLS + EDGE_COLS]
    .agg(["median", "mean", "std"])
)
summary.columns = ["__".join(c) for c in summary.columns]
summary.T


## 3 · Node-count distributions

Density histograms on a `log1p` axis (first cell) and box plots on a symlog axis with outliers hidden (second cell), for the principal node types — files, scopes, types, methods, constructors, fields, parameters. The log transform is necessary because real repositories produce heavy-tailed counts (a few enormous projects coexist with many tiny ones); a linear axis would crush everything against the y-axis. Look for *whole-distribution shifts* between Java and C#, not just outlier moves — that is what indicates a real language-level size difference.

**References:**

- Louridas, Spinellis & Vlachos 2008, *Power laws in software* (ACM TOSEM 18(1), Article 2)
- Concas, Marchesi, Pinna & Serra 2007, *Power-Laws in a Large Object-Oriented Software System* (IEEE TSE 33(10):687–708)
- Herraiz & Hassan 2011, *Beyond Lines of Code: Do We Need More Complexity Metrics?* in Oram & Wilson (eds.), *Making Software* (O'Reilly, ch. 8)

In [ ]:
PLOT_NODE_COLS = [
    "n_folder", "n_file", "n_scope", "n_type",
    "n_operation_method", "n_operation_constructor",
    "n_variable_field", "n_variable_param",
]
labels = {
    "n_folder": "Folders",
    "n_file": "Files", "n_scope": "Scopes (packages/namespaces)",
    "n_type": "Types (classes)", "n_operation_method": "Methods",
    "n_operation_constructor": "Constructors",
    "n_variable_field": "Fields", "n_variable_param": "Parameters",
}

n_cols = len(PLOT_NODE_COLS)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, col in zip(axes, PLOT_NODE_COLS):
    for lang, grp in stats.groupby("lang"):
        vals = grp[col].dropna()
        if vals.empty:
            continue
        ax.hist(
            np.log1p(vals), bins=40, alpha=0.65,
            color=LANG_PALETTE.get(lang, "grey"), label=lang, density=True,
        )
    ax.set_title(labels.get(col, col), fontsize=10)
    ax.set_xlabel("log₁₊(count)")
    ax.set_ylabel("density")
    ax.legend(fontsize=8)

# Hide spare axes.
for ax in axes[n_cols:]:
    ax.set_visible(False)

fig.suptitle("Node-count distributions (log₁₊ scale) by language", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Box plots for direct median comparison.
fig, axes = plt.subplots(1, len(PLOT_NODE_COLS), figsize=(20, 5))
for ax, col in zip(axes, PLOT_NODE_COLS):
    data = [stats.loc[stats["lang"] == lg, col].dropna().values
            for lg in stats["lang"].unique()]
    lang_names = list(stats["lang"].unique())
    ax.boxplot(data, labels=lang_names, showfliers=False, patch_artist=True,
               boxprops=dict(facecolor="#ccddee", alpha=0.7))
    ax.set_title(labels.get(col, col), fontsize=9)
    ax.set_yscale("symlog")
    ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
fig.suptitle("Node counts (symlog scale) — box plots, outliers hidden", fontsize=12)
plt.tight_layout()
plt.show()


## 4 · Edge-label matrix heatmap

For each language, the heatmap shows two rows per structural edge label: the **coverage rate** (fraction of repos in which the edge appears at least once) and the `log1p`-median edge count per repo. The follow-up bar chart isolates the four "reliability-signal" edges (`requires`, `specializes`, `instantiates`, `typed`) — labels whose absence usually means the extractor failed rather than that the language lacks the construct. Together they tell us which edge labels are trustworthy enough to use in any cross-language statistic later.

**References:**

- Kalliamvakou et al. 2016, *An in-depth study of the promises and perils of mining GitHub* (EMSE 21(5):2035–2071) — for the discipline of reporting extractor reliability before any cross-repo claim
- Valverde & Solé 2005, *Hierarchical Small Worlds in Software Architecture* (Advances in Complex Systems 8(1)) — for treating typed-edge dependency graphs as the structural model of a code base

In [ ]:
# Structural edge labels in schema order.
STRUCT_LABELS = [
    "includes", "contains", "declares", "encloses",
    "specializes", "encapsulates", "returns", "instantiates",
    "invokes", "uses", "parameterizes", "typed",
]
mat_cols = [c for c in STRUCT_LABELS if c in matrix.columns]

fig, axes = plt.subplots(1, len(stats["lang"].unique()),
                          figsize=(6 * len(stats["lang"].unique()), 6),
                          sharey=True)
if len(stats["lang"].unique()) == 1:
    axes = [axes]

for ax, lang in zip(axes, stats["lang"].unique()):
    sub = matrix[matrix["lang"] == lang][mat_cols]
    # Presence matrix: 1 if any edges, 0 otherwise.
    presence = (sub > 0).astype(int)
    coverage = presence.mean().rename("coverage rate")

    # Median edge count per repo (log scale) as the heatmap value.
    log_median = np.log1p(sub.median())
    hm_data = pd.DataFrame({
        "coverage": coverage,
        "log_median_count": log_median,
    })

    sns.heatmap(
        hm_data.T, ax=ax,
        annot=True, fmt=".2f", cmap="YlOrRd",
        linewidths=0.5, linecolor="white",
        cbar_kws={"shrink": 0.6},
    )
    ax.set_title(f"{lang}  (n={len(sub)})", fontsize=11)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right", fontsize=9)

fig.suptitle("Edge coverage rate & log₁₊(median count) per language", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Coverage flags — proportion of repos where each reliability-signal edge appears.
COVERAGE_COLS = [c for c in stats.columns if c.startswith("coverage_") and not "requires" in c]
if COVERAGE_COLS:
    cov = stats.groupby("lang")[COVERAGE_COLS].mean().T
    cov.index = [c.replace("coverage_", "") for c in cov.index]

    ax = cov.plot(kind="bar", figsize=(9, 4), color=[LANG_PALETTE.get(l, "grey")
                                                       for l in cov.columns])
    ax.set_ylabel("Proportion of repos with ≥1 edge")
    ax.set_title("Extractor coverage — reliability-signal edge labels")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(title="lang")
    plt.tight_layout()
    plt.show()
else:
    print("No coverage_ columns found.")


## 5 · Structural metrics — fan-in / fan-out / arity

Density histograms of mean call fan-out, call fan-in, method arity, methods/class, fields/class, and field-access fan-in, per repo, split by language. These six metrics span the three classical OO-design dimensions of the CK suite — *coupling* (fan-in/out), *size* (methods, fields per class), and *interface complexity* (arity). A systematic shift between languages here — e.g. consistently higher arity in C# — points to an idiomatic API style difference, not a sampling artefact.

**References:**

- Chidamber & Kemerer 1994, *A Metrics Suite for Object-Oriented Design* (IEEE TSE 20(6):476–493) — the canonical CK metrics (WMC, RFC, CBO, NOC, LCOM)
- Briand, Daly & Wüst 1999, *A Unified Framework for Coupling Measurement in Object-Oriented Systems* (IEEE TSE 25(1):91–121)

In [ ]:
FANOUT_METRICS = {
    "call_fanout_mean":   "Call fan-out (mean/repo)",
    "call_fanin_mean":    "Call fan-in (mean/repo)",
    "arity_mean":         "Arity (mean/repo)",
    "methods_per_class_mean": "Methods per class (mean)",
    "fields_per_class_mean":  "Fields per class (mean)",
    "field_uses_fanin_mean":  "Field accesses / field (mean)",
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for ax, (col, title) in zip(axes, FANOUT_METRICS.items()):
    if col not in stats.columns:
        ax.set_visible(False)
        continue
    for lang, grp in stats.groupby("lang"):
        vals = grp[col].dropna()
        if vals.empty:
            continue
        ax.hist(
            vals, bins=35, alpha=0.65,
            color=LANG_PALETTE.get(lang, "grey"), label=lang, density=True,
        )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("value")
    ax.set_ylabel("density")
    ax.legend(fontsize=8)

fig.suptitle("Fan-in / fan-out / arity distributions by language", fontsize=13)
plt.tight_layout()
plt.show()


## 6 · Inheritance & hierarchy depth

Histograms of maximum inheritance depth (DIT-equivalent), file-system nesting depth, and scope (package/namespace) depth, with a follow-up table reporting the share of repos in which any inheritance was extracted at all. Null values are *not zeros* — they mean the `specializes` edge type was absent from extraction; we keep them visible rather than imputing, because inheritance-flat projects and extraction-failed projects are different things. DIT is one of the most-validated OO metrics for predicting fault-proneness.

**References:**

- Chidamber & Kemerer 1994, *A Metrics Suite for Object-Oriented Design* (IEEE TSE 20(6):476–493) — defines DIT and NOC
- Basili, Briand & Melo 1996, *A Validation of Object-Oriented Design Metrics as Quality Indicators* (IEEE TSE 22(10):751–761)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

depth_cols = {
    "inh_depth_max":   "Max inheritance depth",
    "fs_depth_max":    "Max file-system depth",
    "scope_depth_max": "Max scope/package depth",
}
for ax, (col, title) in zip(axes, depth_cols.items()):
    if col not in stats.columns:
        ax.set_visible(False)
        continue
    sub = stats[["lang", col]].dropna()
    if sub.empty:
        ax.set_visible(False)
        continue

    # Separate into "present" (not-null) vs "absent" (null = edge type missing).
    # For inh_depth_max: null means specializes edges never appear → flat.
    for lang, grp in sub.groupby("lang"):
        vals = grp[col].dropna()
        ax.hist(vals, bins=max(1, int(vals.max()) + 1) if len(vals) else 1,
                alpha=0.65, color=LANG_PALETTE.get(lang, "grey"), label=lang,
                align="left", rwidth=0.7)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("depth (null = edge type absent)")
    ax.set_ylabel("repos")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.legend(fontsize=8)

fig.suptitle("Hierarchy depth distributions", fontsize=12)
plt.tight_layout()
plt.show()

# Summary table: % repos with inheritance edges present, mean depth among those.
inh = stats[["lang", "inh_depth_max", "inh_children_max"]].copy()
inh["has_inh"] = inh["inh_depth_max"].notna() & (inh["inh_depth_max"] > 0)
print(inh.groupby("lang").agg(
    repos=("lang", "size"),
    pct_with_inheritance=("has_inh", "mean"),
    inh_depth_mean=("inh_depth_max", "mean"),
    inh_depth_max=("inh_depth_max", "max"),
    inh_children_max=("inh_children_max", "max"),
).round(3))


## 7 · Method complexity — statements & Halstead

Density histograms (log scale) of per-method statement counts (mean / p90 / max) and of the three Halstead summaries (volume, effort, estimated bugs); the second cell is a scatter of repo size (`n_type`) against method complexity (`stmts_mean`) to test whether complexity grows with project size. Halstead **volume** captures information content from operator/operand vocabulary; **effort** = volume × difficulty and is a better proxy for cognitive load than raw LOC. Reading: a flat scatter would say complexity is invariant to size; an upward trend would support the "big projects also have bigger methods" claim.

**References:**

- Halstead 1977, *Elements of Software Science* (Elsevier) — vocabulary-based complexity
- McCabe 1976, *A Complexity Measure* (IEEE TSE SE-2(4):308–320) — the cyclomatic counterpart often co-reported
- Herraiz & Hassan 2011, *Beyond Lines of Code: Do We Need More Complexity Metrics?* in *Making Software* (O'Reilly) — empirical evidence that LOC and Halstead metrics are highly correlated in practice

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

COMPLEXITY_COLS = {
    "stmts_mean":              "Statements per method (mean)",
    "stmts_p90":               "Statements p90",
    "stmts_max":               "Statements max",
    "halstead_volume_mean":    "Halstead volume (mean)",
    "halstead_effort_mean":    "Halstead effort (mean)",
    "halstead_estimatedBugs_mean": "Estimated bugs (mean)",
}

for ax, (col, title) in zip(axes, COMPLEXITY_COLS.items()):
    if col not in stats.columns:
        ax.set_visible(False)
        continue
    for lang, grp in stats.groupby("lang"):
        vals = grp[col].dropna()
        if vals.empty:
            continue
        ax.hist(
            np.log1p(vals), bins=40, alpha=0.65,
            color=LANG_PALETTE.get(lang, "grey"), label=lang, density=True,
        )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("log₁₊(value)")
    ax.set_ylabel("density")
    ax.legend(fontsize=8)

fig.suptitle("Method complexity distributions (log₁₊ scale)", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Scatter: repo size (n_type) vs complexity proxy (stmts_mean).
fig, ax = plt.subplots(figsize=(9, 6))
for lang, grp in stats.groupby("lang"):
    ax.scatter(
        np.log1p(grp["n_type"]),
        np.log1p(grp["stmts_mean"].fillna(0)),
        alpha=0.5, s=25,
        color=LANG_PALETTE.get(lang, "grey"), label=lang,
    )
ax.set_xlabel("log₁₊(n_type)  — repo size proxy")
ax.set_ylabel("log₁₊(stmts_mean)  — method complexity")
ax.set_title("Repo size vs. method complexity")
ax.legend()
plt.tight_layout()
plt.show()


## 8 · Cross-metric correlation heatmap (per language)

Spearman rank correlation matrix between the main structural and complexity metrics, computed separately for each language and masked to the lower triangle. Spearman is rank-based and therefore robust to the heavy-tailed, non-normal shape of software metrics — Pearson on raw counts would be dominated by the largest projects. Reading: tightly-coupled blocks (|ρ| > 0.7) reveal redundant metrics; sign flips between Java and C# matrices suggest a genuinely different design regime, not just noise.

**References:**

- Spearman 1904, *The Proof and Measurement of Association between Two Things* (American Journal of Psychology 15(1):72–101)
- Gil & Lalouche 2017, *On the correlation between size and metric validity* (Empirical Software Engineering 22(5):2585–2611) — the classic warning that many OO metrics are largely proxies for size

In [ ]:
CORR_COLS = [
    "n_type", "n_operation_method", "n_variable_field",
    "methods_per_class_mean", "fields_per_class_mean",
    "call_fanout_mean", "call_fanin_mean", "arity_mean",
    "inh_depth_max", "stmts_mean",
    "halstead_volume_mean", "halstead_effort_mean",
]
corr_cols_present = [c for c in CORR_COLS if c in stats.columns]

langs = stats["lang"].unique()
fig, axes = plt.subplots(1, len(langs), figsize=(10 * len(langs), 9))
if len(langs) == 1:
    axes = [axes]

for ax, lang in zip(axes, langs):
    sub = stats[stats["lang"] == lang][corr_cols_present].dropna(how="all")
    corr = sub.corr(method="spearman")
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr, ax=ax, mask=mask, annot=True, fmt=".2f",
        cmap="RdBu_r", vmin=-1, vmax=1, linewidths=0.4,
        cbar_kws={"shrink": 0.7},
        xticklabels=[c.replace("_mean","").replace("halstead_","h_") for c in corr.columns],
        yticklabels=[c.replace("_mean","").replace("halstead_","h_") for c in corr.index],
    )
    ax.set_title(f"Spearman correlations — {lang}  (n={len(sub)})", fontsize=11)
    ax.tick_params(axis="x", rotation=40, labelsize=8)
    ax.tick_params(axis="y", rotation=0, labelsize=8)

plt.suptitle("Cross-metric Spearman correlations by language", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 9 · Language comparison — Mann-Whitney U tests

For every numeric metric, runs a two-sided Mann–Whitney U test between every language pair, reports the rank-biserial effect size *r*, applies a Bonferroni correction over the full test family, and ranks the top 20 by |r| in the bar chart. We use Mann–Whitney rather than Student's *t* because the distributions in §3/§7 violate normality, and we lean on rank-biserial *r* rather than p-values because at *n* = 571 nearly every metric clears p < 0.05 — only effect size separates *real* from *trivially detectable* differences. The chart's red/blue indicates which language is larger on each metric.

**References:**

- Mann & Whitney 1947, *On a Test of Whether One of Two Random Variables is Stochastically Larger than the Other* (Annals of Mathematical Statistics 18(1):50–60)
- Kerby 2014, *The Simple Difference Formula: An Approach to Teaching Nonparametric Correlation* (Comprehensive Psychology 3, Article 1) — for rank-biserial *r*
- Arcuri & Briand 2014, *A Hitchhiker's Guide to Statistical Tests for Assessing Randomized Algorithms in Software Engineering* (Software Testing, Verification and Reliability 24(3):219–250) — the SE-specific reference for choosing non-parametric tests + effect sizes over raw p-values

In [ ]:
# Pairwise Mann-Whitney U (non-parametric, no normality assumption) for every
# numeric metric, with Bonferroni correction.
langs = list(stats["lang"].unique())
pairs = list(itertools.combinations(langs, 2))
numeric_cols = stats.select_dtypes(include="number").columns.difference(
    ["n_project"]  # constant; not interesting
)

results = []
for col in numeric_cols:
    for la, lb in pairs:
        a = stats.loc[stats["lang"] == la, col].dropna()
        b = stats.loc[stats["lang"] == lb, col].dropna()
        if len(a) < 3 or len(b) < 3:
            continue
        u, p = spstats.mannwhitneyu(a, b, alternative="two-sided")
        # Effect size: rank-biserial correlation
        n1, n2 = len(a), len(b)
        r = 1 - (2 * u) / (n1 * n2)
        results.append({"metric": col, "lang_a": la, "lang_b": lb,
                         "U": u, "p_raw": p, "r": r,
                         "median_a": a.median(), "median_b": b.median()})

mw = pd.DataFrame(results)
if mw.empty:
    print("Need ≥2 languages with ≥3 repos each for comparison.")
else:
    # Bonferroni correction.
    mw["p_bonf"] = (mw["p_raw"] * len(mw)).clip(upper=1.0)
    mw["significant"] = mw["p_bonf"] < 0.05
    sig = mw[mw["significant"]].sort_values("r", key=abs, ascending=False)
    print(f"Significant after Bonferroni correction: {len(sig)} / {len(mw)}")
    sig[["metric", "lang_a", "lang_b", "median_a", "median_b",
         "r", "p_bonf"]].head(30)


In [ ]:
# Effect-size bar chart for significant metrics.
if not mw.empty and mw["significant"].any():
    sig_top = mw[mw["significant"]].sort_values("r", key=abs, ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(10, max(4, len(sig_top) * 0.4)))
    colors = [("steelblue" if r > 0 else "tomato") for r in sig_top["r"]]
    ax.barh(sig_top["metric"], sig_top["r"], color=colors, alpha=0.8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Rank-biserial correlation r  (positive = lang_a > lang_b)")
    ax.set_title(f"Significant language differences — effect size"
                 f"({sig_top['lang_a'].iloc[0]} vs {sig_top['lang_b'].iloc[0]}, "
                 f"Bonferroni p<0.05)")
    plt.tight_layout()
    plt.show()


## 10 · Outlier & reliability review

Lists the top-10 repos on each extreme dimension (size, fan-out, methods/class, statements/method, inheritance depth) and the repos with **zero coverage** on every reliability-signal edge. The first list flags candidates worth visiting manually for ground-truth checking and for deciding whether to clip the upper tail; the second flags repos where extraction almost certainly failed — typically documentation-only repositories (e.g. `awesome-*`, interview-question collections) or projects with no buildable compilation context. Both lists feed directly into the §11 sampling decision.

**References:**

- Kalliamvakou et al. 2016, *An in-depth study of the promises and perils of mining GitHub* (EMSE 21(5):2035–2071) — for the practice of filtering "non-software" repositories
- Tukey 1977, *Exploratory Data Analysis* (Addison-Wesley) — for the boxplot/extreme-tail inspection idiom

In [ ]:
# Repos in the extreme tails on several dimensions — useful for manual inspection
# and for deciding Study #5 sampling.

OUTLIER_COLS = {
    "n_type":            "Types",
    "call_fanout_max":   "Max call fan-out",
    "methods_per_class_max": "Max methods/class",
    "stmts_max":         "Max statements/method",
    "inh_depth_max":     "Max inheritance depth",
}

for col, label in OUTLIER_COLS.items():
    if col not in stats.columns:
        continue
    sub = stats[["repo", "subrepo", "lang", col]].dropna()
    if sub.empty:
        continue
    print(f"\n── {label} ── top 10 ──────────────────────────────────")
    print(sub.nlargest(10, col)[["lang", "repo", "subrepo", col]].to_string(index=False))


In [ ]:
# Repos with zero coverage on ALL reliability-signal edges — may be empty extractions.
COV_COLS = [c for c in stats.columns if c.startswith("coverage_")]
if COV_COLS:
    zero_cov = stats[stats[COV_COLS].sum(axis=1) == 0][["lang", "repo", "subrepo"] + COV_COLS]
    print(f"Repos with zero coverage on all reliability-signal edges: {len(zero_cov)}")
    if len(zero_cov) <= 30:
        print(zero_cov.to_string(index=False))
    else:
        print(zero_cov.groupby("lang").size().rename("count").to_frame())


## 11 · Study #5 — candidate sampling frame

Identifies a stratified sample of repos suitable for LLM role-stereotype probing.
Adjust `N_PER_STRATUM` and the complexity thresholds to taste.

Stratifies the eligible repos by language × size tertile (small/medium/large by `n_type`), drops repos with zero classes or zero methods (extractor likely failed, see §10), then draws *N* per stratum with a fixed random seed. Stratified sampling is necessary because an unstratified random draw from a power-law-distributed population would be dominated by tiny repos and miss the large ones entirely. The fixed seed (`random_state=42`) makes the sample frame reproducible across re-runs; the second cell exports it to `study5_sample.csv`.

**References:**

- Cochran 1977, *Sampling Techniques* (3rd ed., John Wiley & Sons) — canonical reference for stratified random sampling
- Nagappan, Zimmermann & Bird 2013, *Diversity in Software Engineering Research* (ESEC/FSE 2013, pp. 466–476) — for representativeness when sampling GitHub repositories

In [ ]:
N_PER_STRATUM = 10   # repos per (lang × size_stratum) cell

# Size proxy: n_type (number of classes).  Split into small/medium/large tertiles.
stats["size_stratum"] = stats.groupby("lang")["n_type"].transform(
    lambda x: pd.qcut(x, q=3, labels=["small", "medium", "large"], duplicates="drop")
)

# Exclude repos with no Type nodes (extractor likely failed) and no methods.
eligible = stats[
    (stats["n_type"] > 0) &
    (stats["n_operation_method"] > 0)
].copy()

sample = (
    eligible
    .groupby(["lang", "size_stratum"], observed=True)
    .apply(lambda g: g.sample(min(N_PER_STRATUM, len(g)), random_state=42))
    .reset_index(drop=False)
)

print(f"Sample size: {len(sample)}")
print(sample.groupby(["lang", "size_stratum"]).size().rename("n").to_frame())
sample[["lang", "size_stratum", "repo", "subrepo", "n_type",
        "n_operation_method", "stmts_mean", "inh_depth_max"]]


In [ ]:
# Export sample list for the LLM probing pipeline.
SAMPLE_OUT = Path("study5_sample.csv")
sample[["lang", "repo", "subrepo", "size_stratum"]].to_csv(SAMPLE_OUT, index=False)
print(f"Saved → {SAMPLE_OUT}")


## 12 · Summary table

Compact median ± std table over the headline metrics, formatted for direct lift into the paper's "structural characterisation" section. Median (rather than mean) is reported because of the heavy right tails seen in §3; std (rather than IQR) is kept for compatibility with how most empirical SE papers report dispersion, and *n* is included per language so readers can spot missing values caused by absent edge types (e.g. inheritance depth in repos without `specializes` edges).

**References:**

- Wilkinson & the APA Task Force on Statistical Inference 1999, *Statistical methods in psychology journals: Guidelines and explanations* (American Psychologist 54(8):594–604)
- Jedlitschka, Ciolkowski & Pfahl 2008, *Reporting Experiments in Software Engineering*, in *Guide to Advanced Empirical Software Engineering* (Springer, pp. 201–228)

In [ ]:
# Compact per-language table of key metrics (median ± std).
SUMMARY_METRICS = [
    ("n_file",                  "Files"),
    ("n_type",                  "Types"),
    ("n_operation_method",      "Methods"),
    ("methods_per_class_mean",  "Methods/class"),
    ("fields_per_class_mean",   "Fields/class"),
    ("call_fanout_mean",        "Call fan-out"),
    ("call_fanin_mean",         "Call fan-in"),
    ("arity_mean",              "Arity"),
    ("inh_depth_max",           "Inh. depth"),
    ("stmts_mean",              "Stmts/method"),
    ("halstead_volume_mean",    "Halstead vol."),
    ("halstead_effort_mean",    "Halstead effort"),
]

rows = []
for col, label in SUMMARY_METRICS:
    if col not in stats.columns:
        continue
    row = {"metric": label}
    for lang in stats["lang"].unique():
        vals = stats.loc[stats["lang"] == lang, col].dropna()
        row[f"{lang}_median"] = vals.median()
        row[f"{lang}_std"]    = vals.std()
        row[f"{lang}_n"]      = len(vals)
    rows.append(row)

summary_tbl = pd.DataFrame(rows).set_index("metric")
summary_tbl


## 12 · Scope & Folder Fanout Analysis

This section examines four structural fanout measures produced by `compute_graph_stats.py`:

| Column prefix | Edge / relationship | Interpretation |
|---|---|---|
| `scope_to_scope_fanout_*` | `Scope -[encloses]→ Scope` | Number of **child scopes** (sub-packages/namespaces) per scope |
| `scope_to_type_fanout_*` | `Scope -[encloses]→ Type` | Number of **types** declared directly inside each scope |
| `folder_to_folder_fanout_*` | `Folder -[contains]→ Folder` | Number of **sub-folders** per folder (physical directory tree) |
| `folder_to_file_fanout_*` | `Folder -[contains]→ File` | Number of **files** per folder |

Each measure is captured as **mean, p90, and max** per repository.  
We compare distributions across languages and inspect tail behaviour (p90 vs max).

**References:**

- Lungu et al. 2006, *Package Patterns in Object-Oriented Software Systems* (WCRE)
- Bieman & Kang 1998, *Measuring Design-Level Cohesion* (IEEE TSE 24(2))

In [ ]:
# ── Descriptive statistics for the four fanout groups ──────────────────────
FANOUT_GROUPS = [
    ("scope_to_scope_fanout",   "Scope→Scope  (child scopes)"),
    ("scope_to_type_fanout",    "Scope→Type   (types per scope)"),
    ("folder_to_folder_fanout", "Folder→Folder (sub-folders)"),
    ("folder_to_file_fanout",   "Folder→File  (files per folder)"),
]

STATS_SUFFIXES = ["mean", "p90", "max"]

for prefix, label in FANOUT_GROUPS:
    cols = [f"{prefix}_{s}" for s in STATS_SUFFIXES]
    available = [c for c in cols if c in stats.columns]
    if not available:
        print(f"[SKIP] No columns found for {prefix}")
        continue
    print(f"\n=== {label} ===")
    display(
        stats.groupby("lang")[available]
        .agg(["median", "mean", "std", lambda x: x.quantile(0.9)])
        .rename(columns={"<lambda_0>": "q90"})
    )

### 12.1 · Violin plots per metric

For each of the four fanout groups we plot **mean**, **p90**, and **max**
side-by-side per language.  Axes are on a `log1p` scale to handle the
heavy-tailed distributions typical of software structure metrics.

In [ ]:
for prefix, label in FANOUT_GROUPS:
    cols = [f"{prefix}_{s}" for s in STATS_SUFFIXES]
    available = [c for c in cols if c in stats.columns]
    if not available:
        continue

    fig, axes = plt.subplots(1, len(available), figsize=(5 * len(available), 5), sharey=False)
    if len(available) == 1:
        axes = [axes]

    for ax, suffix in zip(axes, STATS_SUFFIXES):
        col = f"{prefix}_{suffix}"
        if col not in stats.columns:
            ax.set_visible(False)
            continue
        sub = stats[["lang", col]].dropna().copy()
        sub[col] = np.log1p(sub[col])
        palette = {l: LANG_PALETTE.get(l, "#999999") for l in sub["lang"].unique()}
        sns.violinplot(
            data=sub, x="lang", y=col, palette=palette,
            inner="box", cut=0, ax=ax
        )
        ax.set_title(f"{suffix}  (log1p)")
        ax.set_xlabel("")
        ax.set_ylabel(f"log1p({suffix} fanout)")

    fig.suptitle(label, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

### 12.2 · ECDF of per-repo mean fanout

Empirical CDFs on `*_mean` columns give a language-level view of the *typical*
fanout distribution, unaffected by extreme individual outliers.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (prefix, label) in zip(axes, FANOUT_GROUPS):
    col = f"{prefix}_mean"
    if col not in stats.columns:
        ax.set_visible(False)
        continue
    for lang, grp in stats.groupby("lang"):
        vals = grp[col].dropna().sort_values()
        ecdf = np.arange(1, len(vals) + 1) / len(vals)
        ax.plot(np.log1p(vals), ecdf,
                label=lang, color=LANG_PALETTE.get(lang, "#999999"), lw=2)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("log1p(mean fanout)")
    ax.set_ylabel("ECDF")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("ECDF of per-repo mean fanout (log1p scale)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 12.3 · Tail ratio: max / p90

A ratio >> 1 signals 'hotspot' scopes or folders that dwarf the 90th-percentile  
value — an early warning sign of God-package / God-folder antipatterns.

In [ ]:
tail_rows = []
for prefix, label in FANOUT_GROUPS:
    col_max = f"{prefix}_max"
    col_p90 = f"{prefix}_p90"
    if col_max not in stats.columns or col_p90 not in stats.columns:
        continue
    tmp = stats[["lang", col_max, col_p90]].dropna().copy()
    tmp["tail_ratio"] = tmp[col_max] / tmp[col_p90].replace(0, np.nan)
    for lang, grp in tmp.groupby("lang"):
        tail_rows.append({
            "Metric": label,
            "lang": lang,
            "median tail ratio": round(grp["tail_ratio"].median(), 3),
            "p90 tail ratio":    round(grp["tail_ratio"].quantile(0.9), 3),
        })

tail_df = pd.DataFrame(tail_rows).set_index(["Metric", "lang"])
display(tail_df)

### 12.4 · Logical density vs Physical density

Scatter of `scope_to_type_fanout_mean` vs `folder_to_file_fanout_mean` per repo —
do denser package namespaces correlate with denser physical directories?

In [ ]:
xcol = "scope_to_type_fanout_mean"
ycol = "folder_to_file_fanout_mean"

if xcol in stats.columns and ycol in stats.columns:
    sub = stats[["lang", xcol, ycol]].dropna()
    fig, ax = plt.subplots(figsize=(8, 6))
    for lang, grp in sub.groupby("lang"):
        ax.scatter(
            np.log1p(grp[xcol]), np.log1p(grp[ycol]),
            label=lang, color=LANG_PALETTE.get(lang, "#999999"),
            alpha=0.5, s=30
        )
        r, p = spstats.pearsonr(np.log1p(grp[xcol]), np.log1p(grp[ycol]))
        print(f"{lang}: Pearson r = {r:.3f}  (p = {p:.3e}, n = {len(grp)})")
    ax.set_xlabel("log1p(scope→type mean fanout)")
    ax.set_ylabel("log1p(folder→file mean fanout)")
    ax.set_title("Logical density vs Physical density per repo")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] One or both fanout columns not found in stats.")

### 12.5 · Mann–Whitney U: Java vs C# per fanout

Non-parametric significance tests for language-level fanout differences.

In [ ]:
langs = sorted(stats["lang"].unique())
if len(langs) >= 2:
    lang_a, lang_b = langs[0], langs[1]
    mw_rows = []
    for prefix, label in FANOUT_GROUPS:
        for suffix in STATS_SUFFIXES:
            col = f"{prefix}_{suffix}"
            if col not in stats.columns:
                continue
            a = stats.loc[stats["lang"] == lang_a, col].dropna()
            b = stats.loc[stats["lang"] == lang_b, col].dropna()
            if len(a) < 5 or len(b) < 5:
                continue
            stat, p = spstats.mannwhitneyu(a, b, alternative="two-sided")
            mw_rows.append({
                "Metric": label,
                "Statistic": suffix,
                f"{lang_a} median": round(a.median(), 3),
                f"{lang_b} median": round(b.median(), 3),
                "MW U": int(stat),
                "p-value": round(p, 5),
                "sig": "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns")),
            })
    mw_df = pd.DataFrame(mw_rows).set_index(["Metric", "Statistic"])
    display(mw_df)
else:
    print("Need at least two languages to run MW test.")

### 12.6 · Structural metrics style — fanout histograms

Density histograms of mean scope and folder fanouts, per repo, split by language. This mirrors the analysis style of the classic structural metrics, showing the overall distribution shape and language-specific tendencies.

In [ ]:
FANOUT_HIST_METRICS = {
    "scope_to_scope_fanout_mean": "Scope→Scope (mean/repo)",
    "scope_to_type_fanout_mean":  "Scope→Type (mean/repo)",
    "folder_to_folder_fanout_mean": "Folder→Folder (mean/repo)",
    "folder_to_file_fanout_mean":   "Folder→File (mean/repo)",
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for ax, (col, title) in zip(axes, FANOUT_HIST_METRICS.items()):
    if col not in stats.columns:
        ax.set_visible(False)
        continue
    for lang, grp in stats.groupby("lang"):
        vals = grp[col].dropna()
        if vals.empty:
            continue
        ax.hist(
            vals, bins=35, alpha=0.65,
            color=LANG_PALETTE.get(lang, "grey"), label=lang, density=True,
        )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("value")
    ax.set_ylabel("density")
    ax.legend(fontsize=8)

fig.suptitle("Scope and Folder Fanout distributions by language", fontsize=13)
plt.tight_layout()
plt.show()


## 13 · Dependency profile distribution

Each Type (class/interface) may carry a **dependency profile** that captures its coupling role within the package structure:

| Profile | Definition | Architectural reading |
|---|---|---|
| **inbound** | Used by types in other packages; does not use them | Foundational / stable |
| **outbound** | Uses other packages; not used by them | Leaf / consumer |
| **transit** | Both uses and is used by other packages | Bridge / hub |
| **hidden** | Neither uses nor is used by other packages | Isolated / dead |

The extractor encodes this as `Type -[implements]→ Category(dp:*)` edges. Not all graphs carry this annotation; `pct_type_classified` is a coverage signal.

We aggregate profiles at two levels: (1) **repo-wide type counts** — the raw inbound/outbound/transit/hidden mix across all classified types; (2) **package (Scope) level** — the 4-component fraction vector per package, then rolled up as mean/max across all qualifying packages. The Shannon entropy of that vector summarises whether packages have pure roles or are architecturally blurred.

**References:**

- Martin 2003, *Agile Software Development: Principles, Patterns, and Practices* (Prentice Hall) — defines the Instability metric (*I* = Ce / (Ca + Ce)) and associates inbound ↔ stable, outbound ↔ instable, transit ↔ balanced
- Briand, Daly & Wüst 1999, *A Unified Framework for Coupling Measurement in Object-Oriented Systems* (IEEE TSE 25(1):91–121)
- Valverde & Solé 2005, *Hierarchical Small Worlds in Software Architecture* (Advances in Complex Systems 8(1))


In [ ]:
PROFILES       = ["inbound", "outbound", "transit", "hidden"]
DP_TYPE_COLS   = [f"n_type_{p}" for p in PROFILES]
PROFILE_COLORS = {
    "inbound":  "#2196F3",
    "outbound": "#FF9800",
    "transit":  "#9C27B0",
    "hidden":   "#9E9E9E",
}

if not all(c in stats.columns for c in DP_TYPE_COLS):
    print("No dependency profile columns found — re-run compute_graph_stats.py.")
else:
    stats["_dp_n_classified"] = stats[DP_TYPE_COLS].sum(axis=1)
    stats["_has_dp"]          = stats["_dp_n_classified"] > 0

    cov = (
        stats.groupby("lang")
        .agg(repos=("lang", "size"),
             pct_with_profiles=("_has_dp", "mean"),
             median_pct_classified=("pct_type_classified", "median"))
        .round(3)
    )
    print(cov)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    ax = axes[0]
    langs = list(cov.index)
    ax.bar(langs, cov["pct_with_profiles"],
           color=[LANG_PALETTE.get(l, "grey") for l in langs], alpha=0.8)
    ax.set_ylabel("Fraction of repos")
    ax.set_title("Repos with any dependency profile data")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylim(0, 1.05)

    ax = axes[1]
    has_dp = stats[stats["_has_dp"]]
    med_classified = has_dp.groupby("lang")["pct_type_classified"].median()
    ax.bar(med_classified.index, med_classified.values,
           color=[LANG_PALETTE.get(l, "grey") for l in med_classified.index], alpha=0.8)
    ax.set_ylabel("Median fraction of types classified")
    ax.set_title("Type-level coverage\n(repos with profile data only)")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylim(0, 1.05)

    plt.suptitle("Dependency profile extractor coverage", fontsize=12)
    plt.tight_layout()
    plt.show()


### 13.1 · Repo-level type profile composition

For each repo the classified types are split into four fractions (inbound + outbound + transit + hidden = 1). The stacked bar shows the **mean composition** across repos per language, giving the characteristic profile mix of a typical project in that language. The grouped box plot shows how much variability there is in each fraction across repos — wide boxes indicate that individual projects differ substantially from the average.


In [ ]:
if "_has_dp" not in stats.columns or not stats["_has_dp"].any():
    print("[SKIP] No repos with dependency profile data.")
else:
    dp = stats[stats["_has_dp"]].copy()
    for p in PROFILES:
        dp[f"_frac_{p}"] = dp[f"n_type_{p}"] / dp["_dp_n_classified"]

    mean_comp = dp.groupby("lang")[[f"_frac_{p}" for p in PROFILES]].mean()
    mean_comp.columns = PROFILES

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: stacked bar — mean type composition per language.
    ax = axes[0]
    bottom = np.zeros(len(mean_comp))
    for p in PROFILES:
        ax.bar(mean_comp.index, mean_comp[p], bottom=bottom,
               label=p, color=PROFILE_COLORS[p], alpha=0.85)
        for i, (_, val) in enumerate(mean_comp[p].items()):
            if val > 0.05:
                ax.text(i, bottom[i] + val / 2, f"{val:.0%}",
                        ha="center", va="center", fontsize=9,
                        color="white", fontweight="bold")
        bottom += mean_comp[p].values
    ax.set_ylabel("Mean fraction of classified types")
    ax.set_title("Average type profile composition\n(fraction of classified types per repo)")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(title="profile", loc="upper right")

    # Right: grouped box plot — distribution of per-repo fractions across profiles and languages.
    ax = axes[1]
    long_rows = []
    for p in PROFILES:
        sub = dp[["lang", f"_frac_{p}"]].dropna().rename(columns={f"_frac_{p}": "fraction"})
        sub["profile"] = p
        long_rows.append(sub)
    long_df = pd.concat(long_rows, ignore_index=True)

    sns.boxplot(data=long_df, x="profile", y="fraction", hue="lang",
                palette=LANG_PALETTE, ax=ax, showfliers=False)
    ax.set_ylabel("Per-repo fraction of classified types")
    ax.set_title("Profile fraction distribution across repos\n(outliers hidden)")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(title="lang")

    plt.suptitle("Repo-level type profile composition", fontsize=13)
    plt.tight_layout()
    plt.show()


### 13.2 · Package-level profile fractions

`pkg_pct_{p}_mean` is the mean, across all qualifying packages in a repo, of the fraction of their classified types that carry profile *p*. A high mean inbound fraction says most packages in the repo are predominantly foundational; a high transit mean flags many architectural hubs. The `_max` companion records the most concentrated package — a max of 1.0 means at least one pure single-profile package exists.


In [ ]:
PKG_MEAN_COLS = {p: f"pkg_pct_{p}_mean" for p in PROFILES}
available_pkg = {p: c for p, c in PKG_MEAN_COLS.items() if c in stats.columns}

if not available_pkg:
    print("[SKIP] No pkg_pct_* columns found.")
else:
    pkg_has_data = stats[stats[[c for c in available_pkg.values()]].notna().any(axis=1)]

    fig, axes = plt.subplots(1, len(available_pkg),
                              figsize=(4.5 * len(available_pkg), 5),
                              sharey=True)
    if len(available_pkg) == 1:
        axes = [axes]

    for ax, (p, col) in zip(axes, available_pkg.items()):
        sub = pkg_has_data[["lang", col]].dropna()
        palette = {l: LANG_PALETTE.get(l, "#999999") for l in sub["lang"].unique()}
        sns.violinplot(data=sub, x="lang", y=col,
                       palette=palette, inner="box", cut=0, ax=ax)
        ax.set_title(p, fontsize=11, fontweight="bold",
                     color=PROFILE_COLORS.get(p, "black"))
        ax.set_xlabel("")
        ax.set_ylabel("Mean package fraction" if p == PROFILES[0] else "")
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

    fig.suptitle("Per-repo mean package profile fractions (violin)", fontsize=13)
    plt.tight_layout()
    plt.show()

    # Median of per-repo max — how extreme can the most concentrated package get?
    PKG_MAX_COLS  = {p: f"pkg_pct_{p}_max" for p in PROFILES}
    available_max = {p: c for p, c in PKG_MAX_COLS.items() if c in stats.columns}
    if available_max:
        print("\nMedian of pkg_pct_{p}_max across repos (most concentrated package per repo):")
        display(
            pkg_has_data.groupby("lang")[[c for c in available_max.values()]]
            .median()
            .rename(columns={v: k for k, v in available_max.items()})
            .round(3)
        )


### 13.3 · Dominant-role distribution & package entropy

**Dominant role**: for each qualifying package we pick the profile with the highest fraction of its classified types (ties broken alphabetically). The column `pct_pkg_dominant_{p}` records what fraction of all qualifying packages in a repo are dominated by profile *p*. A repo where most packages are *inbound*-dominant reads like a library — lots of stable, foundational components. One where most are *outbound*-dominant reads like an application — lots of leaf consumers.

**Package entropy**: the Shannon entropy of the 4-component profile vector per package (range 0–log₂4 ≈ 2 bits) measures whether packages have clear, single-profile identities (low entropy) or are architecturally blurred junk-drawers (high entropy). `pkg_profile_entropy_mean` is the mean across all qualifying packages in the repo. The scatter of entropy against repo size checks whether larger projects inevitably produce more mixed packages.


In [ ]:
DOM_COLS      = {p: f"pct_pkg_dominant_{p}" for p in PROFILES}
available_dom = {p: c for p, c in DOM_COLS.items() if c in stats.columns}

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── Left: stacked bar — mean dominant-role distribution per language ─────
ax = axes[0]
if available_dom:
    dom_has  = stats[stats[[c for c in available_dom.values()]].notna().any(axis=1)]
    dom_mean = dom_has.groupby("lang")[[c for c in available_dom.values()]].mean()
    dom_mean.columns = list(available_dom.keys())

    bottom = np.zeros(len(dom_mean))
    for p in PROFILES:
        if p not in dom_mean.columns:
            continue
        ax.bar(dom_mean.index, dom_mean[p], bottom=bottom,
               label=p, color=PROFILE_COLORS[p], alpha=0.85)
        for i, (_, val) in enumerate(dom_mean[p].items()):
            if val > 0.06:
                ax.text(i, bottom[i] + val / 2, f"{val:.0%}",
                        ha="center", va="center", fontsize=8,
                        color="white", fontweight="bold")
        bottom += dom_mean[p].values
    ax.set_ylabel("Mean fraction of qualifying packages")
    ax.set_title("Dominant-role distribution\n(avg across repos per language)")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(title="dominant profile")
else:
    ax.set_visible(False)

# ── Middle: entropy violin ───────────────────────────────────────────────
ax = axes[1]
if "pkg_profile_entropy_mean" in stats.columns:
    ent_sub = stats[["lang", "pkg_profile_entropy_mean"]].dropna()
    palette  = {l: LANG_PALETTE.get(l, "#999999") for l in ent_sub["lang"].unique()}
    sns.violinplot(data=ent_sub, x="lang", y="pkg_profile_entropy_mean",
                   palette=palette, inner="box", cut=0, ax=ax)
    ax.axhline(np.log2(4), color="tomato", linestyle="--", linewidth=1.2,
               label=f"max entropy ({np.log2(4):.2f} bits)")
    ax.axhline(0, color="seagreen", linestyle="--", linewidth=1.2,
               label="pure (0 bits)")
    ax.set_title("Package profile entropy\n(mean per repo)")
    ax.set_xlabel("")
    ax.set_ylabel("Shannon entropy (bits)")
    ax.legend(fontsize=8)
else:
    ax.set_visible(False)

# ── Right: scatter repo size vs entropy ─────────────────────────────────
ax = axes[2]
if "pkg_profile_entropy_mean" in stats.columns:
    for lang, grp in stats.groupby("lang"):
        sub = grp[["n_type", "pkg_profile_entropy_mean"]].dropna()
        if sub.empty:
            continue
        ax.scatter(np.log1p(sub["n_type"]), sub["pkg_profile_entropy_mean"],
                   alpha=0.45, s=18,
                   color=LANG_PALETTE.get(lang, "grey"), label=lang)
    ax.set_xlabel("log₁₊(n_type)  — repo size")
    ax.set_ylabel("pkg_profile_entropy_mean (bits)")
    ax.set_title("Repo size vs. package role entropy")
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.set_visible(False)

plt.suptitle("Dominant-role architecture & package profile entropy", fontsize=13)
plt.tight_layout()
plt.show()
